In [1]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
## Data Ingestion-- From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

/home/udesh_kohli/Code/LLM_Learnings/Langchain/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
URL = "https://docs.langchain.com/oss/python/deepagents/quickstart"
loader = WebBaseLoader(URL)

In [4]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/deepagents/quickstart', 'title': 'Quickstart - Docs by LangChain', 'description': 'Build your first deep agent in minutes', 'language': 'en'}, page_content='Quickstart - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationGet startedQuickstartDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedQuickstartCustomizationComparisonChangelogCore capabilitiesModelsOverviewBackendsSubagentsHuman-in-the-loopLong-term memorySkillsSandboxesStreamingFrontendOverviewSubagent StreamingTodo ListProtocolsAgent Client Protocol (ACP)Command line interfaceUse the CLIModel providersConfigurationMCP ToolsOn this pagePrerequisitesStep 1: Install dependenciesStep 2: Set up your API keysStep 3: Create a search toolStep 4: Create a deep agentStep 5: Run the agentHow does it work?ExamplesStreamingNext stepsGet starte

In [5]:
## Load Data --> Docs --> Divide out text into chunks --> text --> vectors --> Vector Embeddings -- > Vector store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
documents = text_splitter.split_documents(docs)

In [8]:
type(documents)


list

Embedding the documents

In [9]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [10]:
from langchain_community.vectorstores import FAISS
vectorstoredb = FAISS.from_documents(documents, embeddings)

In [11]:
vectorstoredb

In [20]:
#import chatopenAI
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o") #Model Names: As of 2026, you generally want to use gpt-4o (Omni) or gpt-4-turbo, as they are faster and more capable than the original gpt-4.
print(llm)

profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x7f89649d4850> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7f89649d7820> root_client=<openai.OpenAI object at 0x7f89649677f0> root_async_client=<openai.AsyncOpenAI object at 0x7f89649d61a0> model_name='gpt-4o' model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True


In [ ]:
#query from a vector db
query = "How to Set up your API keys"
result = vectorstoredb.similarity_search(query)
result[0].page_content

'This guide uses Tavily as an example search provider, but you can substitute any search API (e.g., DuckDuckGo, SerpAPI, Brave Search).\n\u200bStep 2: Set up your API keys\n Anthropic OpenAI Google OpenRouter Fireworks Baseten OllamaCopyexport ANTHROPIC_API_KEY="your-api-key"\nexport TAVILY_API_KEY="your-tavily-api-key"\nCopyexport OPENAI_API_KEY="your-api-key"\nexport TAVILY_API_KEY="your-tavily-api-key"\nCopyexport GOOGLE_API_KEY="your-api-key"\nexport TAVILY_API_KEY="your-tavily-api-key"\nCopyexport OPENROUTER_API_KEY="your-api-key"\nexport TAVILY_API_KEY="your-tavily-api-key"\nCopyexport FIREWORKS_API_KEY="your-api-key"\nexport TAVILY_API_KEY="your-tavily-api-key"\nCopyexport BASETEN_API_KEY="your-api-key"\nexport TAVILY_API_KEY="your-tavily-api-key"\nCopy# Local: Ollama must be running (https://ollama.com)\n# Cloud: Set your Ollama API key for hosted inference\nexport OLLAMA_API_KEY="your-api-key"\nexport TAVILY_API_KEY="your-tavily-api-key"'

In [21]:
## Retrieval chain, document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(

    """
Answer the following question based only on the provided context:
<context>
{context}
</context>

"""
)
document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, cli

In [27]:
## Input ----> Retriever --> Vectorstoredb


retriever = vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [25]:
retrieval_chain 

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f8964dd4730>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n'), additional_kwargs={})])
            | ChatOp

In [ ]:
## Get the response of the LLM
reponse = retrieval_chain.invoke({"input": "How to Customize your agent?"})

In [35]:
reponse

{'input': 'How to Set up your API keys?Step 2: Set up your API keys?',
 'context': [Document(id='5f628399-e472-4090-8ab5-388c8e3661d2', metadata={'source': 'https://docs.langchain.com/oss/python/deepagents/quickstart', 'title': 'Quickstart - Docs by LangChain', 'description': 'Build your first deep agent in minutes', 'language': 'en'}, page_content='This guide uses Tavily as an example search provider, but you can substitute any search API (e.g., DuckDuckGo, SerpAPI, Brave Search).\n\u200bStep 2: Set up your API keys\n Anthropic OpenAI Google OpenRouter Fireworks Baseten OllamaCopyexport ANTHROPIC_API_KEY="your-api-key"\nexport TAVILY_API_KEY="your-tavily-api-key"\nCopyexport OPENAI_API_KEY="your-api-key"\nexport TAVILY_API_KEY="your-tavily-api-key"\nCopyexport GOOGLE_API_KEY="your-api-key"\nexport TAVILY_API_KEY="your-tavily-api-key"\nCopyexport OPENROUTER_API_KEY="your-api-key"\nexport TAVILY_API_KEY="your-tavily-api-key"\nCopyexport FIREWORKS_API_KEY="your-api-key"\nexport TAVILY_AP